# Week 3: Clean Complaint Text and Prepare Target Labels

This notebook prepares the 2024 CFPB model-development dataset for Week 3 exploratory data analysis and future supervised NLP modeling.

Scope rules for this notebook:

- Use only `data/raw/cfpb_complaints_2024_raw.csv`.
- Do not load or use the 2025 holdout dataset.
- Use `complaint_what_happened` as the original complaint narrative text column.
- Use `product` as the target label.
- Create a local-only cleaned CSV under `data/processed/`.
- Do not train models, score models, create confusion matrices, or make final evaluation claims.

## 1. Imports and Project Paths

The paths below are explicit so this notebook loads the 2024 model-development file only. The 2025 file is named only to confirm it remains separate and is not read by this notebook.

In [12]:
from pathlib import Path
import re
import subprocess

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)


def find_project_root(start_path: Path) -> Path:
    """Find the repository root from either the project root or notebooks folder."""
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".gitignore").exists() and (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not find project root with .gitignore and data/raw.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_2024_PATH = RAW_DIR / "cfpb_complaints_2024_raw.csv"
HOLDOUT_2025_PATH = RAW_DIR / "cfpb_complaints_2025_raw.csv"
OUTPUT_PATH = PROCESSED_DIR / "cfpb_complaints_2024_cleaned.csv"

TEXT_COLUMN = "complaint_what_happened"
TARGET_COLUMN = "product"
SHORT_COMPLAINT_WORD_THRESHOLD = 5

print(f"Project root: {PROJECT_ROOT}")
print(f"Selected raw input: {RAW_2024_PATH.relative_to(PROJECT_ROOT)}")
print(f"Future holdout file, not loaded here: {HOLDOUT_2025_PATH.relative_to(PROJECT_ROOT)}")

Project root: E:\MGA\ITEC6740\Final-Project\financial-complaint-auto-routing-nlp
Selected raw input: data\raw\cfpb_complaints_2024_raw.csv
Future holdout file, not loaded here: data\raw\cfpb_complaints_2025_raw.csv


## 2. Confirm the 2024-Only Dataset Selection

This section confirms that the selected input file is the 2024 raw CFPB dataset. The 2025 holdout file may exist locally, but it is not loaded or used for Week 3 cleaning.

In [13]:
selected_input_files = [RAW_2024_PATH]

if not RAW_2024_PATH.exists():
    raise FileNotFoundError(
        f"Expected 2024 raw CFPB dataset was not found: {RAW_2024_PATH.relative_to(PROJECT_ROOT)}"
    )

assert RAW_2024_PATH.parent == RAW_DIR, "The selected raw file must come from data/raw/."
assert RAW_2024_PATH.name == "cfpb_complaints_2024_raw.csv", "Only the 2024 raw dataset should be loaded."
assert all("2024" in path.name for path in selected_input_files), "Every selected input file must be a 2024 file."
assert not any("2025" in path.name for path in selected_input_files), "The 2025 holdout file must not be selected."

print("Confirmed selected input files:")
for path in selected_input_files:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

print(f"2025 holdout file exists locally: {HOLDOUT_2025_PATH.exists()}")
print("2025 holdout status: separate future out-of-time holdout; not loaded in this notebook.")

Confirmed selected input files:
- data\raw\cfpb_complaints_2024_raw.csv
2025 holdout file exists locally: True
2025 holdout status: separate future out-of-time holdout; not loaded in this notebook.


## 3. Load the 2024 Raw Dataset

Only the 2024 raw CSV is read here. The required columns are validated before any cleaning is performed.

In [14]:
raw_df = pd.read_csv(RAW_2024_PATH, low_memory=False)
original_row_count = len(raw_df)

required_columns = {TEXT_COLUMN, TARGET_COLUMN}
missing_required_columns = sorted(required_columns.difference(raw_df.columns))
if missing_required_columns:
    raise KeyError(f"Missing required columns in 2024 raw data: {missing_required_columns}")

print(f"Loaded rows from 2024 raw dataset: {original_row_count:,}")
print(f"Loaded columns: {len(raw_df.columns):,}")
print(f"Input text column: {TEXT_COLUMN}")
print(f"Target label column: {TARGET_COLUMN}")

display(pd.DataFrame({"column": raw_df.columns, "dtype": raw_df.dtypes.astype(str).values}))

Loaded rows from 2024 raw dataset: 50,000
Loaded columns: 17
Input text column: complaint_what_happened
Target label column: product


,column,dtype
0,product,str
1,complaint_what_happened,str
2,date_sent_to_company,str
3,issue,str
4,sub_product,str
5,zip_code,str
6,tags,str
7,has_narrative,bool
8,complaint_id,int64
9,timely,str


## 4. Check Missing Text and Product Labels

Rows without complaint narrative text or without a product label cannot be used for supervised product classification. Blank strings are treated as missing.

In [15]:
text_missing_mask = raw_df[TEXT_COLUMN].astype("string").str.strip().fillna("").eq("")
product_missing_mask = raw_df[TARGET_COLUMN].astype("string").str.strip().fillna("").eq("")
rows_to_remove_mask = text_missing_mask | product_missing_mask

missing_text_count = int(text_missing_mask.sum())
missing_product_count = int(product_missing_mask.sum())
missing_either_count = int(rows_to_remove_mask.sum())

missing_summary = pd.DataFrame(
    {
        "check": ["missing complaint text", "missing product label", "missing either required field"],
        "row_count": [missing_text_count, missing_product_count, missing_either_count],
        "percent_of_raw": [
            missing_text_count / original_row_count * 100,
            missing_product_count / original_row_count * 100,
            missing_either_count / original_row_count * 100,
        ],
    }
)
missing_summary["percent_of_raw"] = missing_summary["percent_of_raw"].round(2)
display(missing_summary)

,check,row_count,percent_of_raw
0,missing complaint text,0,0.0
1,missing product label,0,0.0
2,missing either required field,0,0.0


## 5. Create Clean Modeling Columns

The cleaned dataset keeps only two modeling columns: `complaint_text` and `product`. Text cleaning is intentionally light so the complaint narratives remain natural enough for future TF-IDF and DistilBERT modeling. The cleaning removes obvious URLs, normalizes whitespace, and strips leading or trailing spaces. It does not lowercase, stem, remove stop words, or remove punctuation.

In [16]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r"\s+")


def clean_complaint_text(value: object) -> str:
    """Apply light cleaning while preserving natural complaint language."""
    text = "" if pd.isna(value) else str(value)
    text = URL_PATTERN.sub(" ", text)
    text = WHITESPACE_PATTERN.sub(" ", text)
    return text.strip()


cleaned_df = raw_df.loc[~rows_to_remove_mask, [TEXT_COLUMN, TARGET_COLUMN]].copy()
cleaned_df["complaint_text"] = cleaned_df[TEXT_COLUMN].map(clean_complaint_text)
cleaned_df["product"] = cleaned_df[TARGET_COLUMN].astype("string").str.strip()
cleaned_df = cleaned_df[["complaint_text", "product"]]

empty_after_cleaning_mask = cleaned_df["complaint_text"].str.len().eq(0) | cleaned_df["product"].astype("string").str.len().eq(0)
rows_empty_after_cleaning = int(empty_after_cleaning_mask.sum())

if rows_empty_after_cleaning:
    cleaned_df = cleaned_df.loc[~empty_after_cleaning_mask].copy()

cleaned_df = cleaned_df.reset_index(drop=True)
cleaned_row_count = len(cleaned_df)
rows_removed = original_row_count - cleaned_row_count

print(f"Rows retained after required-field filtering and light text cleaning: {cleaned_row_count:,}")
print(f"Rows removed total: {rows_removed:,}")
print(f"Rows empty after URL/whitespace cleaning: {rows_empty_after_cleaning:,}")
print(f"Cleaned modeling columns: {list(cleaned_df.columns)}")

display(cleaned_df.dtypes.astype(str).rename("dtype").reset_index().rename(columns={"index": "column"}))

Rows retained after required-field filtering and light text cleaning: 50,000
Rows removed total: 0
Rows empty after URL/whitespace cleaning: 0
Cleaned modeling columns: ['complaint_text', 'product']


,column,dtype
0,complaint_text,str
1,product,string


## 6. Text-Length Checks

Character length and word count help identify unusual records before modeling. Very short complaints are counted for awareness, but they are not removed automatically in this Week 3 cleaning step.

In [17]:
eda_text_df = cleaned_df.assign(
    character_length=cleaned_df["complaint_text"].str.len(),
    word_count=cleaned_df["complaint_text"].str.split().str.len(),
)

very_short_complaint_count = int(eda_text_df["word_count"].lt(SHORT_COMPLAINT_WORD_THRESHOLD).sum())

text_length_summary = eda_text_df[["character_length", "word_count"]].describe().T.round(2)
display(text_length_summary)

print(
    f"Very short complaints, fewer than {SHORT_COMPLAINT_WORD_THRESHOLD} words: "
    f"{very_short_complaint_count:,}"
)

,count,mean,std,min,25%,50%,75%,max
character_length,50000.0,1051.42,1455.82,10.0,305.0,679.0,1275.0,32550.0
word_count,50000.0,182.13,250.61,1.0,53.0,117.0,224.0,5853.0


Very short complaints, fewer than 5 words: 62


## 7. Label-Readiness Checks

The `product` column is the supervised target label. This section checks the number of classes and the product class distribution. These checks are descriptive only and do not train or evaluate a model.

In [18]:
product_label_counts = cleaned_df["product"].value_counts().rename_axis("product").reset_index(name="complaint_count")
number_of_product_classes = cleaned_df["product"].nunique()

class_distribution_table = product_label_counts.copy()
class_distribution_table["class_percent"] = (
    class_distribution_table["complaint_count"] / cleaned_row_count * 100
).round(2)

print(f"Number of unique product labels: {number_of_product_classes:,}")
display(class_distribution_table)

Number of unique product labels: 11


,product,complaint_count,class_percent
0,Credit reporting or other personal consumer re...,36128,72.26
1,Debt collection,5357,10.71
2,Credit card,2735,5.47
3,Checking or savings account,2146,4.29
4,Mortgage,887,1.77
5,"Money transfer, virtual currency, or money ser...",725,1.45
6,Student loan,628,1.26
7,Vehicle loan or lease,590,1.18
8,"Payday loan, title loan, personal loan, or adv...",363,0.73
9,Prepaid card,317,0.63


## 8. Duplicate Narrative Check

Duplicates are counted here for data-quality awareness only. No duplicate rows are removed in this Week 3 notebook. Duplicate-handling decisions will be considered later during modeling and train/validation/test splitting to avoid possible train-test leakage.

In [19]:
exact_duplicate_pair_rows = int(cleaned_df.duplicated(subset=["complaint_text", "product"]).sum())

duplicate_pair_groups = (
    cleaned_df.groupby(["complaint_text", "product"])
    .size()
    .gt(1)
    .sum()
)

duplicate_text_rows = int(cleaned_df.duplicated(subset=["complaint_text"]).sum())
duplicate_text_values = int(cleaned_df["complaint_text"].value_counts().gt(1).sum())

text_product_counts = cleaned_df.groupby("complaint_text")["product"].nunique()
texts_with_multiple_products = int(text_product_counts.gt(1).sum())

duplicate_narrative_summary = pd.DataFrame(
    {
        "check": [
            "exact duplicate complaint_text + product rows after first occurrence",
            "exact duplicate complaint_text + product groups",
            "duplicate complaint_text rows after first occurrence",
            "duplicate complaint_text values",
            "complaint_text values linked to more than one product label",
        ],
        "count": [
            exact_duplicate_pair_rows,
            int(duplicate_pair_groups),
            duplicate_text_rows,
            duplicate_text_values,
            texts_with_multiple_products,
        ],
    }
)

display(duplicate_narrative_summary)

,check,count
0,exact duplicate complaint_text + product rows ...,16006
1,exact duplicate complaint_text + product groups,4189
2,duplicate complaint_text rows after first occu...,16082
3,duplicate complaint_text values,4158
4,complaint_text values linked to more than one ...,74


## 9. Save the Local Cleaned Dataset

The cleaned CSV is saved locally for future modeling notebooks. It should remain ignored by Git and should not be committed.

In [20]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved cleaned local dataset to: {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved columns: {list(cleaned_df.columns)}")
print(f"Saved rows: {len(cleaned_df):,}")

Saved cleaned local dataset to: data\processed\cfpb_complaints_2024_cleaned.csv
Saved columns: ['complaint_text', 'product']
Saved rows: 50,000


## 10. Git Safety Check: Confirm Processed CSV Is Ignored


In [21]:
gitignore_path = PROJECT_ROOT / ".gitignore"
gitignore_text = gitignore_path.read_text(encoding="utf-8")

processed_rule_present = "data/processed/*" in gitignore_text or "data/processed/" in gitignore_text
csv_rule_present = "*.csv" in gitignore_text

relative_output_path = OUTPUT_PATH.relative_to(PROJECT_ROOT).as_posix()
check_ignore = subprocess.run(
    ["git", "check-ignore", relative_output_path],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
git_confirms_output_ignored = check_ignore.returncode == 0

print(f".gitignore contains data/processed ignore rule: {processed_rule_present}")
print(f".gitignore contains *.csv ignore rule: {csv_rule_present}")
print(f"git check-ignore confirms output CSV is ignored: {git_confirms_output_ignored}")

if not processed_rule_present or not csv_rule_present or not git_confirms_output_ignored:
    raise RuntimeError("Processed CSV ignore check failed. Review .gitignore before committing.")

.gitignore contains data/processed ignore rule: True
.gitignore contains *.csv ignore rule: True
git check-ignore confirms output CSV is ignored: True


## 11. Final Week 3 Summary

The summary below documents the row counts, required-field removals, class count, and output path for the cleaned 2024 modeling dataset.

In [22]:
final_summary = pd.DataFrame(
    {
        "metric": [
            "original_row_count",
            "cleaned_row_count",
            "rows_removed",
            "missing_text_count",
            "missing_product_count",
            "number_of_product_classes",
            "output_path",
        ],
        "value": [
            f"{original_row_count:,}",
            f"{cleaned_row_count:,}",
            f"{rows_removed:,}",
            f"{missing_text_count:,}",
            f"{missing_product_count:,}",
            f"{number_of_product_classes:,}",
            relative_output_path,
        ],
    }
)

display(final_summary)

,metric,value
0,original_row_count,"50,000"
1,cleaned_row_count,"50,000"
2,rows_removed,0
3,missing_text_count,0
4,missing_product_count,0
5,number_of_product_classes,11
6,output_path,data/processed/cfpb_complaints_2024_cleaned.csv
